# Window Soft Argmax Inference - PF-Willow Dataset

This notebook performs semantic correspondence inference using the Window Soft Argmax (WSA) strategy on the PF-Willow dataset.

## What this notebook does:
- Tests all backbone models (DINOv2, DINOv3, SAM) on PF-Willow dataset
- Uses Window Soft Argmax prediction strategy for more precise localization
- Supports both pretrained and fine-tuned model weights
- Reports PCK@α metrics for α ∈ {0.05, 0.1, 0.2}

## About PF-Willow:
- Willow ObjectClass dataset for semantic correspondence
- Contains pairs of objects with significant intra-class variations
- Known for challenging deformation and appearance changes
- Fixed set of 10 keypoints per object pair

## WSA Strategy:
- Creates a window around the argmax peak
- Applies softmax with temperature scaling
- Computes weighted average of coordinates within the window
- Generally provides better localization than simple argmax

## Configuration options:
- Change `BACKBONE` to 'dinov2', 'dinov3', or 'sam'
- Change `USE_FINETUNED` to True/False for fine-tuned vs pretrained weights

## Expected runtime:
- ~5-10 minutes per backbone (PF-Willow has fewer samples)

In [ ]:
%pip install torchmetrics                # ONLY FOR FIRST EXECUTION
%pip install git+https://github.com/facebookresearch/segment-anything.git

from google.colab import drive
import os

REPO_URL = "https://github.com/AML-Semantic-Correspondence/Semantic_Correspondence.git"

# 2. Clone/Pull the Code (access to logic)
print("\n Setting up repository...")

# First ensure we're in a safe directory
%cd /content

# Clean up any existing problematic directories
if os.path.exists('/content/Semantic_Correspondence'):
    print("Removing existing Semantic_Correspondence directory...")
    !rm -rf /content/Semantic_Correspondence

if os.path.exists('/content/semantic-correspondence'):
    print("Removing existing semantic-correspondence directory...")
    !rm -rf /content/semantic-correspondence

# Clone the repository
print("Cloning repository fresh...")
try:
    !git clone {REPO_URL}
    
    # The repo will be cloned as 'Semantic_Correspondence', let's rename it for consistency
    if os.path.exists('/content/Semantic_Correspondence'):
        !mv /content/Semantic_Correspondence /content/semantic-correspondence
        print(" Repository cloned and renamed successfully")
    else:
        print(" Repository clone failed")
except Exception as e:
    print(f" Error during clone: {e}")

# Mount drive and extract datasets
drive.mount("/content/drive", force_remount=True)
!tar -xzf "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/SPair-71k.tar.gz"
!unzip -o -q "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/PF-dataset-PASCAL.zip"
!unzip -o -q "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/PF-dataset.zip"

# Add the repository to path
%cd /content/semantic-correspondence
import sys
sys.path.append('/content/semantic-correspondence')

# Import inference functions
from src.inference.run_evaluation import run_evaluation
from src.inference.wsa import wsa_strategy

# ===== CONFIGURATION =====
# Change these parameters to test different combinations
BACKBONE = 'dinov2'     # Options: 'dinov2', 'dinov3', 'sam'
USE_FINETUNED = False   # True for fine-tuned weights, False for pretrained

# Path to fine-tuned weights (adjust if needed)
FINETUNED_WEIGHTS_PATH = "/content/drive/MyDrive/AML-Semantic-Correspondence/weights/best_model.pth"

weights_path = None
model_type = "Pretrained"

if USE_FINETUNED:
    if os.path.exists(FINETUNED_WEIGHTS_PATH):
        weights_path = FINETUNED_WEIGHTS_PATH
        model_type = "Fine-tuned"
    else:
        print(f"\n Warning: Fine-tuned weights not found at {FINETUNED_WEIGHTS_PATH}")
        print("Please run the corresponding training notebook first to generate fine-tuned weights.")
        print("Falling back to pretrained weights...")

print(f"\n Running WSA Inference - PF-Willow")
print(f" Configuration:")
print(f"   - Backbone: {BACKBONE}")
print(f"   - Dataset: pf-willow")
print(f"   - Strategy: Window Soft Argmax (WSA)")
print(f"   - Model: {model_type}")

# Run evaluation
print("\n Starting evaluation...")
# Temporarily change directory back to /content where PF-dataset is located
%cd /content/
run_evaluation(
    backbone=BACKBONE,
    dataset_var='pf-willow',
    split='test',  # PF-Willow only has test split
    prediction_method=wsa_strategy,
    weights_path=weights_path
)
# Change back to the repository directory after evaluation
%cd /content/semantic-correspondence


print("\n Evaluation completed!")
print("\n To test other combinations, modify the BACKBONE and USE_FINETUNED variables above and re-run the cell.")
print("\n Note: PF-Willow is known for challenging intra-class variations and deformations.")